# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = "provide your email"
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [3]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Pydantic Models for Validation

In [4]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

### Refactoring | Helper Functions

In [6]:

def clean_llm_json(content: str) -> str:
    """Removes markdown code fences from LLM output string."""
    if "```json" in content:
        return content.split("```json")[1].split("```")[0].strip()
    elif "```" in content:
        return content.split("```")[1].split("```")[0].strip()
    return content.strip()

def get_recipe_ingredients_string(recipe: dict) -> str:
    """Formats a recipe's ingredients list into a single comma-separated string."""
    return ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])

def parse_input_with_default(value: str, default: str) -> str:
    """Returns a default value if the input string is empty or whitespace-only."""
    return value if value and value.strip() else default


# --- Test Case A: Testing an Empty String ---
empty_input = ""
result_empty = parse_input_with_default(empty_input, "random diverse cuisines")
print(f"Test Empty String ('') -> Result: '{result_empty}' (Passed if default used)")

# --- Test Case B: Testing None ---
none_input = None
try:
    result_none = parse_input_with_default(none_input, "random diverse cuisines")
    print(f"Test None -> Result: '{result_none}' (Passed if default used)")
except Exception as e:
    print(f"❌ Failed on None input with error: {type(e).__name__}: {e}")
#-------------------------------------

def parse_json_to_dict(clean_text: str) -> dict:
    """Parses clean text into a standard Python dictionary."""
    return json.loads(clean_text)


def validate_and_build_meal_plan(meal_plan_data: dict) -> MealPlan:
    """Validates dictionary data against the Pydantic MealPlan model."""
    return MealPlan(**meal_plan_data)
def format_success_response(meal_plan: MealPlan, generation_time: float) -> dict:
    """Formats the final successful response payload."""
    return {
        "success": True,
        "meal_plan": meal_plan.model_dump(),
        "generation_time": round(generation_time, 2),
        "error": None
    }

def format_error_response(error_msg: str) -> dict:
    """Returns a standardized error response dictionary."""
    return {"success": False, "meal_plan": None, "error": str(error_msg)}

Test Empty String ('') -> Result: 'random diverse cuisines' (Passed if default used)
Test None -> Result: 'random diverse cuisines' (Passed if default used)


## 5. Meal Generation Logic

### Adding Error Handling

In [ ]:
from openai import AuthenticationError, APIConnectionError, RateLimitError
from pydantic import ValidationError
import smtplib
import json
import time


def create_error_payload(func_name: str, error_type: str, location: str, suggestion: str) -> dict:
    """Core builder: Standardizes the error dictionary structure."""
    return {
        "success": False,
        "meal_plan": None,
        "error": f"[Function: {func_name}] | [Error Type: {error_type}] | [Location: {location}] | Suggestion: {suggestion}"
    }

def handle_app_exception(e: Exception) -> dict:
    """Unified router for OpenAI, JSON, Pydantic, and SMTP errors."""
    if isinstance(e, AuthenticationError):
        return create_error_payload("generate", "AuthenticationError", "OpenAI API Key", "Check your .env file.")
    elif isinstance(e, json.JSONDecodeError):
        return create_error_payload("parse_json_to_dict", "JSONDecodeError", f"Line {e.lineno}, Col {e.colno}", "LLM returned malformed JSON.")
    elif isinstance(e, ValidationError):
        return create_error_payload("validate_and_build_meal_plan", "ValidationError", "Schema Constraints", "LLM violated the required structure.")
    elif isinstance(e, smtplib.SMTPAuthenticationError):
        return create_error_payload("send_meal_plan_email", "SMTPAuthenticationError", "SMTP Server Login", "Check your email App Password in .env.")
    else:
        return create_error_payload("unknown", type(e).__name__, "General Execution", str(e))

In [ ]:

class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.shopping_parser = JsonOutputParser(pydantic_object=ShoppingItem)

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()
        
        try:
            # 1. Parse/normalize inputs using defaults
            themes = parse_input_with_default(theme_requests, "random diverse cuisines")
            prefs = parse_input_with_default(additional_preferences, "none")

            # 2. Invoke the LLM chain
            recipe_chain = MEAL_PLAN_PROMPT | self.llm
            result = recipe_chain.invoke({
                "dietary_preference": dietary_preference,
                "theme_requests": themes,
                "additional_preferences": prefs,
                "format_instructions": self.recipe_parser.get_format_instructions()
            })
            
            # 3. Modular execution steps (Clean -> Parse -> Validate)
            clean_text = clean_llm_json(result.content)                              #Refactored 
            meal_plan_data = parse_json_to_dict(clean_text)                          #Modularized
            meal_plan_obj = validate_and_build_meal_plan(meal_plan_data)             #Modularized
            
            generation_time = time.time() - start_time

            # 4. Return formatted success
            return format_success_response(meal_plan_obj, generation_time)            #Modularized
    
        except Exception as e:
            return handle_app_exception(e)



# Initialize generator
generator = MealPlanGenerator()

## 6. Email Functionality

In [ ]:
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients=get_recipe_ingredients_string(recipe)   #Refactored
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)

#Modularization
 
def build_email_message(recipient_email: str, subject: str, body: str) -> MIMEMultipart:     #Modularization
    """Builds and returns a MIME multipart email container."""
    msg = MIMEMultipart()
    msg['From'] = EMAIL_SENDER
    msg['To'] = recipient_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    return msg

def send_smtp_email(msg: MIMEMultipart) -> None:                                              #Modularization
    """Handles the actual SMTP network transport separately."""
    with smtplib.SMTP('smtp.gmail.com', 587) as server:
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)
        server.send_message(msg)

def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    try:
        body = format_meal_plan_email(meal_plan)
        msg = build_email_message(recipient_email, "Your Weekly Meal Plan", body)
        send_smtp_email(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except Exception as e:
        return {"success": False, "message": f"Failed to send email: {str(e)}"}

## 7. Validation

In [12]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

## 8. Gradio Interface

In [13]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients=get_recipe_ingredients_string(recipe)     #Refactored
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"



#-------------Modularize Streaming output

def format_streaming_header(dietary_pref: str, shopping_list_html: str) -> str:
    """Formats the initial header and shopping list for streaming output."""
    return f"## Weekly {dietary_pref.title()} Meal Plan\n\n{shopping_list_html}\n\n---\n\n## Recipes\n\n"

def format_streaming_footer(generation_time: float, validation_result: dict) -> str:
    """Formats the validation status and generation time footer."""
    status = "All checks passed" if validation_result["passed"] else f"Issues: {', '.join(validation_result['errors'])}"
    return f"---\n*Generated in {generation_time}s | {status}*"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output."""
    output = "Generating your meal plan..."
    yield output
    
    themes = parse_input_with_default(theme_requests, "random") #refactored
    prefs = parse_input_with_default(additional_prefs, "none") #refactored

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=themes,                 
        additional_preferences=prefs       
    )

    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = format_streaming_header(meal_plan['dietary_preference'], format_shopping_list(meal_plan))
    yield output
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    output += format_streaming_footer(result['generation_time'], validation)    
    yield output

#--------------Modularize Input Validation for UI---------------
def is_valid_email(email: str) -> bool:
    """Pure validation function for email formats."""
    return bool(email and "@" in email and "." in email)


def send_email_ui(email, dietary_pref, theme_requests, additional_prefs):
    """Generate and send meal plan via email."""
    if not is_valid_email(email):
        return "Enter a valid email address."
    
    themes = parse_input_with_default(theme_requests, "random") #refactored
    prefs = parse_input_with_default(additional_prefs, "none") #refactored

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=themes,               
        additional_preferences=prefs      
    )

    if not result["success"]:
        return f"Generation failed: {result['error']}"

    email_result = send_meal_plan_email(email, result["meal_plan"])
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, dietary_dropdown, theme_input, additional_input],
        outputs=[email_status]
    )

/home/fatmamerayy/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 9. Launch Application

In [14]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://0784e73ababee24d5b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [ ]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])